## Load Excel files

In [2]:
import pandas as pd
import duckdb
from pathlib import Path

DATA_PATH = Path("../data/raw")

In [202]:
occupancy = pd.read_excel(
    DATA_PATH / "Occupancy Rates.xlsx"
)

travel_mode = pd.read_excel(
    DATA_PATH / "Arrivals (Travel mode).xlsx"
)

length_of_stay = pd.read_excel(
    DATA_PATH / "Arrivals (Sex and Length of Stay).xlsx"
)

## Check files

In [203]:
occupancy.head()
# travel_mode.head()
# length_of_stay.head()

,Month,Quarter,Year,Hotel Tier,Average Room Rate,Average Occupancy Rate
0,Totals,NaN,NaN,NaN,235.390867,0.809048
1,1,Q1,2026.0,Economy,124.971588,0.825163
2,1,Q1,2026.0,Luxury,648.149951,0.784892
3,1,Q1,2026.0,Mid-Tier,201.265330,0.822712
4,1,Q1,2026.0,Upscale,317.171551,0.813956


In [83]:
travel_mode.head()

,Month,Quarter,Year,Sex,Place of Residence,Region of Residence,Mode of Arrival,Visitor Arrivals
0,Totals,NaN,NaN,NaN,NaN,NaN,NaN,242694363
1,1,Q1,2026.0,Female,Australia,OCEANIA,Air,61812
2,1,Q1,2026.0,Female,Australia,OCEANIA,Land,2982
3,1,Q1,2026.0,Female,Australia,OCEANIA,Sea,4369
4,1,Q1,2026.0,Female,Bangladesh,SOUTH ASIA,Air,2543


In [84]:
length_of_stay.head()

,Month,Quarter,Year,Sex,Place of Residence,Region of Residence,Age,Visitor Arrivals,Average Length of Stay
0,Totals,NaN,NaN,NaN,NaN,NaN,NaN,242694363,3.620716
1,1,Q1,2026.0,Female,Australia,OCEANIA,Age 14 & Below,12395,3.452421
2,1,Q1,2026.0,Female,Australia,OCEANIA,Age 15 - 19,5827,3.068722
3,1,Q1,2026.0,Female,Australia,OCEANIA,Age 20 - 24,4700,2.843254
4,1,Q1,2026.0,Female,Australia,OCEANIA,Age 25 - 34,8870,2.627451


In [85]:
print(occupancy.shape)
print(travel_mode.shape)
print(length_of_stay.shape)

(889, 5)
(73404, 8)
(210111, 9)


In [4]:
connection = duckdb.connect()

In [204]:
connection.register("occupancy_raw", occupancy)
connection.register("travel_mode_raw", travel_mode)
connection.register("length_of_stay_raw", length_of_stay)

In [205]:
connection.sql("""
    SELECT *
    FROM occupancy_raw
    LIMIT 10
""").df()

,Month,Quarter,Year,Hotel Tier,Average Room Rate,Average Occupancy Rate
0,Totals,NaN,NaN,NaN,235.390867,0.809048
1,1,Q1,2026.0,Economy,124.971588,0.825163
2,1,Q1,2026.0,Luxury,648.149951,0.784892
3,1,Q1,2026.0,Mid-Tier,201.265330,0.822712
4,1,Q1,2026.0,Upscale,317.171551,0.813956
5,1,Q1,2025.0,Economy,128.244881,0.817618
6,1,Q1,2025.0,Luxury,659.249677,0.824548
7,1,Q1,2025.0,Mid-Tier,204.203603,0.816851
8,1,Q1,2025.0,Upscale,322.625635,0.817968
9,1,Q1,2024.0,Economy,131.840347,0.800770


## Clean tables

In [208]:
# remove null

occupancy_clean = connection.sql("""
    SELECT *
    FROM occupancy_raw
    WHERE "Year" IS NOT NULL
""").df()

occupancy_clean

,Month,Quarter,Year,Hotel Tier,Average Room Rate,Average Occupancy Rate
0,1,Q1,2026.0,Economy,124.971588,0.825163
1,1,Q1,2026.0,Luxury,648.149951,0.784892
2,1,Q1,2026.0,Mid-Tier,201.265330,0.822712
3,1,Q1,2026.0,Upscale,317.171551,0.813956
4,1,Q1,2025.0,Economy,128.244881,0.817618
...,...,...,...,...,...,...
883,12,Q4,2009.0,Upscale,202.267352,0.802312
884,12,Q4,2008.0,Economy,95.412876,0.726959
885,12,Q4,2008.0,Luxury,358.645165,0.674735
886,12,Q4,2008.0,Mid-Tier,171.027245,0.746388


In [209]:
# Convert Month and Year to integer, and Average Occupancy Rate to double for analysis

occupancy_clean = connection.sql("""
    SELECT
        CAST(Month AS INTEGER) AS month,
        Quarter AS quarter,
        CAST(Year AS INTEGER) AS year,
        "Hotel Tier" as hotel_tier,
        CAST("Average Room Rate" AS DOUBLE) AS avg_room_rate,
        CAST("Average Occupancy Rate" AS DOUBLE) AS avg_occupancy_rate,
    MAKE_DATE(
        CAST(Year AS INTEGER),
        CAST(Month AS INTEGER),
        1
    ) AS month_date
    FROM occupancy_clean

""").df()

In [189]:
# check no more null values

connection.sql("""
    SELECT *
    FROM occupancy_clean
    WHERE "Year" IS NULL
""").df()


,Month,Quarter,Year,Hotel Tier,Average Room Rate,Average Occupancy Rate


In [192]:
# check column month_date created properly

connection.sql("""
    SELECT *
    FROM occupancy_clean
""").df()

,month,quarter,year,hotel_tier,avg_room_rate,avg_occupancy_rate,month_date
0,1,Q1,2026,Mid-Tier,201.265330,0.822712,2026-01-01
1,1,Q1,2025,Mid-Tier,204.203603,0.816851,2025-01-01
2,1,Q1,2024,Mid-Tier,205.649811,0.785441,2024-01-01
3,1,Q1,2023,Mid-Tier,201.254754,0.715365,2023-01-01
4,1,Q1,2022,Mid-Tier,120.292583,0.594259,2022-01-01
...,...,...,...,...,...,...,...
217,12,Q4,2012,Mid-Tier,187.646417,0.828232,2012-12-01
218,12,Q4,2011,Mid-Tier,186.720856,0.808359,2011-12-01
219,12,Q4,2010,Mid-Tier,171.143681,0.846869,2010-12-01
220,12,Q4,2009,Mid-Tier,130.958794,0.835431,2009-12-01


In [210]:
connection.register(
    "occupancy_clean",
    occupancy_clean
)

In [13]:
# remove null, modify data types and add month_date to travel_mode

travel_mode_clean = connection.sql("""
    SELECT
        CAST(Month AS INTEGER) AS month,
        Quarter AS quarter,
        CAST(Year AS INTEGER) AS year,
        Sex AS sex,
        "Place of Residence" AS country,
        "Region of Residence" AS region,
        "Mode of Arrival" AS arrival_mode,
        CAST("Visitor Arrivals" AS BIGINT) AS visitor_arrivals,

        MAKE_DATE(
            CAST(Year AS INTEGER),
            CAST(Month AS INTEGER),
            1
        ) AS month_date

    FROM travel_mode_raw

    WHERE Month <> 'Totals'
      AND Year IS NOT NULL
""").df()

In [14]:
# final check

connection.sql("""
    SELECT *
    FROM travel_mode_clean
""").df()

,month,quarter,year,sex,country,region,arrival_mode,visitor_arrivals,month_date
0,1,Q1,2026,Female,Australia,OCEANIA,Air,61812,2026-01-01
1,1,Q1,2026,Female,Australia,OCEANIA,Land,2982,2026-01-01
2,1,Q1,2026,Female,Australia,OCEANIA,Sea,4369,2026-01-01
3,1,Q1,2026,Female,Bangladesh,SOUTH ASIA,Air,2543,2026-01-01
4,1,Q1,2026,Female,Bangladesh,SOUTH ASIA,Land,346,2026-01-01
...,...,...,...,...,...,...,...,...,...
73398,12,Q4,2008,Not Stated,USA,AMERICAS,Land,193,2008-12-01
73399,12,Q4,2008,Not Stated,USA,AMERICAS,Sea,364,2008-12-01
73400,12,Q4,2008,Not Stated,Vietnam,SOUTHEAST ASIA,Air,459,2008-12-01
73401,12,Q4,2008,Not Stated,Vietnam,SOUTHEAST ASIA,Land,139,2008-12-01


In [15]:
connection.register(
    "travel_mode_clean",
    travel_mode_clean
)

In [16]:
#  remove null, convert '-' to 0, modify data types and add month_date to length_of_stay

length_of_stay_clean = connection.sql("""
    SELECT
        CAST(Month AS INTEGER) AS month,
        Quarter AS quarter,
        CAST(Year AS INTEGER) AS year,
        Sex AS sex,
        "Place of Residence" AS country,
        "Region of Residence" AS region,
        Age AS age_group,

        COALESCE(
            TRY_CAST("Visitor Arrivals" AS BIGINT),
            0
        ) AS visitor_arrivals,

        COALESCE(
            TRY_CAST("Average Length of Stay" AS DOUBLE),
            0
        ) AS avg_length_of_stay,

        MAKE_DATE(
            CAST(Year AS INTEGER),
            CAST(Month AS INTEGER),
            1
        ) AS month_date

    FROM length_of_stay_raw

    WHERE Month <> 'Totals'
      AND Year IS NOT NULL
""").df()

In [17]:
# check no more null

connection.sql("""
    SELECT *
    FROM length_of_stay_clean
    WHERE visitor_arrivals IS NULL
       OR avg_length_of_stay IS NULL
""").df()

,month,quarter,year,sex,country,region,age_group,visitor_arrivals,avg_length_of_stay,month_date


In [18]:
# check that '-' converted to 0 successfully

connection.sql("""
    SELECT *
    FROM length_of_stay_clean
    WHERE visitor_arrivals = 0
    OR avg_length_of_stay = 0
""").df()

# note that some rows have avg_length_of_stay <> 0 when visitor_arrivals = 0

,month,quarter,year,sex,country,region,age_group,visitor_arrivals,avg_length_of_stay,month_date
0,1,Q1,2026,Male,Others,OTHERS,Age 35 - 44,1,0.0,2026-01-01
1,1,Q1,2026,Not Stated,Bangladesh,SOUTH ASIA,Age 25 - 34,1,0.0,2026-01-01
2,1,Q1,2026,Not Stated,China,GREATER CHINA,Age 35 - 44,1,0.0,2026-01-01
3,1,Q1,2026,Not Stated,China,GREATER CHINA,Age 55 - 64,0,44.5,2026-01-01
4,1,Q1,2026,Not Stated,Germany,EUROPE,Age 15 - 19,0,1.0,2026-01-01
...,...,...,...,...,...,...,...,...,...,...
8092,12,Q4,2008,Not Stated,Germany,EUROPE,Not Stated,0,7.0,2008-12-01
8093,12,Q4,2008,Not Stated,Iran,WEST ASIA,Age 20 - 24,1,0.0,2008-12-01
8094,12,Q4,2008,Not Stated,Kuwait,WEST ASIA,Age 15 - 19,1,0.0,2008-12-01
8095,12,Q4,2008,Not Stated,Myanmar,SOUTHEAST ASIA,Not Stated,6,0.0,2008-12-01


In [19]:
# convert all rows where avg_length_of_stay to 0 where visitor_arrivals = 0

length_of_stay_clean = connection.sql("""
    SELECT
        * EXCLUDE (avg_length_of_stay),

        CASE
            WHEN visitor_arrivals = 0 THEN 0
            ELSE avg_length_of_stay
        END AS avg_length_of_stay

    FROM length_of_stay_clean
""").df()

In [20]:
# check no more invalid avg_length_of_stay values

connection.sql("""
    SELECT *
    FROM length_of_stay_clean
    WHERE visitor_arrivals = 0
      AND avg_length_of_stay <> 0
""").df()

,month,quarter,year,sex,country,region,age_group,visitor_arrivals,month_date,avg_length_of_stay


In [21]:
connection.register(
    "length_of_stay_clean",
    length_of_stay_clean
)

## Post-cleaning validations

In [22]:
print("Occupancy raw:", len(occupancy))
print("Occupancy clean:", len(occupancy_clean))

print("Travel mode raw:", len(travel_mode))
print("Travel mode clean:", len(travel_mode_clean))

print("Length of stay raw:", len(length_of_stay))
print("Length of stay clean:", len(length_of_stay_clean))

# only one row should have been removed from each table

Occupancy raw: 889
Occupancy clean: 888
Travel mode raw: 73404
Travel mode clean: 73403
Length of stay raw: 210111
Length of stay clean: 210110


In [136]:
# confirm no duplicates in cleaned tables

print(
    "Occupancy duplicates:",
    occupancy_clean.duplicated().sum()
)

print(
    "Travel mode duplicates:",
    travel_mode_clean.duplicated().sum()
)

print(
    "Length of stay duplicates:",
    length_of_stay_clean.duplicated().sum()
)

Occupancy duplicates: 0
Travel mode duplicates: 0
Length of stay duplicates: 0


In [492]:
connection.sql("""
    SELECT
        MIN(avg_occupancy_rate) AS minimum,
        MAX(avg_occupancy_rate) AS maximum
    FROM occupancy_clean
""").df()

# check no value above 1

,minimum,maximum
0,0.160079,0.958891


In [ ]:
connection.sql("""
    SELECT
        MIN(visitor_arrivals) AS minimum,
        MAX(visitor_arrivals) AS maximum
    FROM travel_mode_clean
""").df()

# check no negative value

,minimum,maximum
0,1,200797


In [142]:
connection.sql("""
    SELECT
        l.month,
        l.year,
        l.country,
        l.visitor_arrivals AS length_of_stay_arrivals,
        t.visitor_arrivals AS travel_mode_arrivals

    FROM length_of_stay_clean l

    INNER JOIN travel_mode_clean t
        ON l.month = t.month
        AND l.year = t.year
        AND l.country = t.country

    WHERE l.visitor_arrivals = 0
""").df()

,month,year,country,length_of_stay_arrivals,travel_mode_arrivals
0,1,2026,China,0,120914
1,1,2026,China,0,24877
2,1,2026,China,0,4516
3,1,2026,Germany,0,12839
4,1,2026,Germany,0,495
...,...,...,...,...,...
11139,12,2008,Other Markets in North Asia,0,11
11140,12,2008,Other Markets in North Asia,0,5
11141,12,2008,Other Markets in North Asia,0,27
11142,12,2008,Other Markets in North Asia,0,8


In [143]:
connection.sql("""
    SELECT *
    FROM travel_mode_clean
    WHERE visitor_arrivals < 0
""").df()

# check no negative values

,month,quarter,year,sex,country,region,arrival_mode,visitor_arrivals,month_date


In [23]:
connection.sql("""
    SELECT
        MIN(month_date) AS earliest_date,
        MAX(month_date) AS latest_date
    FROM travel_mode_clean
""").df()

,earliest_date,latest_date
0,2008-01-01,2026-07-01


## (1) Performance Snapshot

## Tourism Performance High-level Analysis

## Observable years are 2008 - 2026, however
# Years 2020 - 2022 have been excluded from all analysis to prevent results from being skewed (COVID-19)

In [ ]:
# In terms of visitor arrivals, what is the average rank of each country for all observable years?

average_country_rank = connection.sql("""
    WITH country_totals AS (
        SELECT
            year,
            country,
            SUM(visitor_arrivals) AS visitor_arrivals
        FROM travel_mode_clean
        WHERE YEAR NOT IN (2020, 2021, 2022)
        GROUP BY year, country
    ),
    ranked AS (
        SELECT
            year,
            country,
            visitor_arrivals,
            ROW_NUMBER() OVER (
                PARTITION BY year
                ORDER BY visitor_arrivals DESC
            ) AS rank
        FROM country_totals
    )
    SELECT
        country,
        ROUND(AVG(rank), 2) AS avg_rank,
        MIN(rank) AS best_rank,
        MAX(rank) AS worst_rank,
        COUNT(DISTINCT year) AS years_observed
    FROM ranked
    GROUP BY country
    ORDER BY avg_rank
""").df()

average_country_rank

# Indonesia, China, Malaysia, India, Australia have top 5 average rank

,country,avg_rank,best_rank,worst_rank,years_observed
0,Indonesia,1.38,1,2,16
1,China,1.63,1,2,16
2,Malaysia,3.44,3,5,16
3,Australia,4.19,3,5,16
4,India,4.38,3,5,16
5,Philippines,6.88,6,9,16
6,Japan,7.00,6,12,16
7,USA,9.00,6,11,16
8,South Korea,9.69,8,12,16
9,UK,10.00,7,12,16


In [ ]:
# Our top 10 markets by average rank (in visitor arrivals) from 2008 - 2026

average_country_rank_top10 = connection.sql("""
    WITH country_totals AS (
        SELECT
            year,
            country,
            SUM(visitor_arrivals) AS visitor_arrivals
        FROM travel_mode_clean
        WHERE YEAR NOT IN (2020, 2021, 2022)
        GROUP BY year, country
    ),
    ranked AS (
        SELECT
            year,
            country,
            visitor_arrivals,
            ROW_NUMBER() OVER (
                PARTITION BY year
                ORDER BY visitor_arrivals DESC
            ) AS rank
        FROM country_totals
    )
    SELECT
        country,
        ROUND(AVG(rank), 2) AS avg_rank,
        MIN(rank) AS best_rank,
        MAX(rank) AS worst_rank,
        COUNT(DISTINCT year) AS years_observed
    FROM ranked
    GROUP BY country
    ORDER BY avg_rank
    LIMIT 10
""").df()

connection.register("average_country_rank_top10", average_country_rank_top10)

average_country_rank_top10


,country,avg_rank,best_rank,worst_rank,years_observed
0,Indonesia,1.38,1,2,16
1,China,1.63,1,2,16
2,Malaysia,3.44,3,5,16
3,Australia,4.19,3,5,16
4,India,4.38,3,5,16
5,Philippines,6.88,6,9,16
6,Japan,7.00,6,12,16
7,USA,9.00,6,11,16
8,South Korea,9.69,8,12,16
9,UK,10.00,7,12,16


In [444]:
# What are the relevant average rankings for each demographic segments (age group, sex) from 2008 - 2026?

avg_demographic_rank = connection.sql("""
    WITH demographic_totals AS (
        SELECT
            year,
            age_group,
            sex,
            SUM(visitor_arrivals) AS demographic_visitors
        FROM length_of_stay_clean
        WHERE YEAR NOT IN (2020, 2021, 2022)
        GROUP BY year, age_group, sex
    ),
    ranked AS (
        SELECT
            year,
            age_group,
            sex,
            demographic_visitors,
            ROW_NUMBER() OVER (
                PARTITION BY year
                ORDER BY demographic_visitors DESC
            ) AS rank
        FROM demographic_totals
    )
    SELECT
        age_group,
        sex,
        ROUND(AVG(rank), 2) AS avg_rank,
        MIN(rank) AS best_rank,
        MAX(rank) AS worst_rank,
        COUNT(DISTINCT year) AS years_observed
    FROM ranked
    GROUP BY age_group, sex
    ORDER BY avg_rank
""").df()

avg_demographic_rank

# top 5 segments consists of males aged 25 - 54 and Females 25 - 44

,age_group,sex,avg_rank,best_rank,worst_rank,years_observed
0,Age 35 - 44,Male,1.56,1,2,16
1,Age 25 - 34,Female,1.63,1,3,16
2,Age 25 - 34,Male,3.06,2,4,16
3,Age 35 - 44,Female,4.25,3,5,16
4,Age 45 - 54,Male,4.50,4,5,16
5,Age 45 - 54,Female,6.00,6,6,16
6,Age 55 - 64,Male,7.50,7,8,16
7,Age 55 - 64,Female,7.50,7,8,16
8,Age 14 & Below,Male,9.13,9,10,16
9,Age 14 & Below,Female,10.13,10,11,16


In [489]:
avg_country_age_rank = connection.sql("""
    WITH country_age_totals AS (
        SELECT
            year,
            country,
            age_group,
            SUM(visitor_arrivals) AS demographic_visitors
        FROM length_of_stay_clean
        WHERE year NOT IN (2020, 2021, 2022)
            AND age_group <> 'Not Stated'
        GROUP BY year, country, age_group
    ),
    ranked AS (
        SELECT
            year,
            country,
            age_group,
            demographic_visitors,
            ROW_NUMBER() OVER (
                PARTITION BY year, country
                ORDER BY demographic_visitors DESC
            ) AS rank
        FROM country_age_totals
    )
    SELECT
        country,
        age_group,
        ROUND(AVG(rank), 2) AS avg_rank,
        MIN(rank) AS best_rank,
        MAX(rank) AS worst_rank,
        COUNT(DISTINCT year) AS years_observed
    FROM ranked
    GROUP BY country, age_group
    ORDER BY country, avg_rank
""").df()

avg_country_age_rank

,country,age_group,avg_rank,best_rank,worst_rank,years_observed
0,Australia,Age 45 - 54,1.31,1,2,16
1,Australia,Age 35 - 44,2.00,1,3,16
2,Australia,Age 25 - 34,3.63,2,5,16
3,Australia,Age 55 - 64,3.63,2,6,16
4,Australia,Age 65 & Above,4.50,3,5,16
...,...,...,...,...,...,...
419,Vietnam,Age 14 & Below,4.56,4,5,16
420,Vietnam,Age 20 - 24,4.94,4,6,16
421,Vietnam,Age 55 - 64,5.50,4,6,16
422,Vietnam,Age 15 - 19,7.25,7,8,16


In [490]:
## Singapore's top 10 tourist markets - Average ranking by demographic segment (age, sex) from 2008 - 2026

avg_country_demographic_rank_top10 = connection.sql("""
    WITH demographic_totals AS (
        SELECT
            year,
            country,
            age_group,
            sex,
            SUM(visitor_arrivals) AS demographic_visitors
        FROM length_of_stay_clean
        WHERE year NOT IN (2020, 2021, 2022)
            AND age_group <> 'Not Stated'
            AND sex <> 'Not Stated'
        GROUP BY year, country, age_group, sex
    ),
    ranked AS (
        SELECT
            year,
            country,
            age_group,
            sex,
            demographic_visitors,
            ROW_NUMBER() OVER (
                PARTITION BY year, country
                ORDER BY demographic_visitors DESC
            ) AS rank
        FROM demographic_totals
    )
    SELECT
        country,
        age_group,
        sex,
        ROUND(AVG(rank), 2) AS avg_rank,
        MIN(rank) AS best_rank,
        MAX(rank) AS worst_rank,
        COUNT(DISTINCT year) AS years_observed
    FROM ranked
    WHERE country IN (SELECT country from average_country_rank_top10)
    GROUP BY country, age_group, sex
    ORDER BY avg_rank ASC
""").df()

avg_country_demographic_rank_top10

,country,age_group,sex,avg_rank,best_rank,worst_rank,years_observed
0,Philippines,Age 25 - 34,Female,1.00,1,1,16
1,USA,Age 45 - 54,Male,1.00,1,1,16
2,India,Age 25 - 34,Male,1.19,1,2,16
3,China,Age 25 - 34,Female,1.25,1,2,16
4,Indonesia,Age 25 - 34,Female,1.31,1,2,16
...,...,...,...,...,...,...,...
155,Indonesia,Age 15 - 19,Male,16.00,16,16,16
156,China,Age 15 - 19,Male,16.00,16,16,16
157,Australia,Age 15 - 19,Male,16.00,16,16,16
158,India,Age 15 - 19,Female,16.00,16,16,16


In [491]:
## What is the average ranking of each age group and sex among these 10 countries

avg_rank_by_age_group_top10 = connection.sql("""
    SELECT
        age_group,
        sex,
        ROUND(AVG(avg_rank), 2) AS avg_rank
    FROM avg_country_demographic_rank_top10
    GROUP BY age_group, sex
    ORDER BY avg_rank ASC
""").df()

avg_rank_by_age_group_top10

## For Singapore's top 10 tourist markets, males aged 25 - 54 and females aged 25 - 44
# are our biggest demographic segments

,age_group,sex,avg_rank
0,Age 35 - 44,Male,2.59
1,Age 25 - 34,Female,3.18
2,Age 25 - 34,Male,3.76
3,Age 45 - 54,Male,3.80
4,Age 35 - 44,Female,5.09
5,Age 45 - 54,Female,6.29
6,Age 55 - 64,Male,6.63
7,Age 55 - 64,Female,7.61
8,Age 14 & Below,Male,9.93
9,Age 65 & Above,Male,10.29


## Potential areas of improvement

For top 10 countries, younger adults and children make up the smallest proportion of tourist arrivals. May wish
to consider exploring how we can better incentivize these groups to visit Singapore.

Probably have to target their parents in the process because these individuals do not have high purchasing power
yet and are more likely to travel with parents. More family-centric events/attractions perhaps?

## (2) Data Meshing

## Cause of Top Markets Performance Deep Dive

Question: Why do Males aged 25 - 54 and females 25 - 44 make up a significant proporrtion of tourists
coming from Singapore's top 10 tourist markets?

Hypothesis: International concerts in Singapore are associated with stronger visitor growth among the key age-sex demographics that make up a significant proportion of visitors from Singapore's top 10 tourist markets.

## Supporting Public Dataset: List of music concerts by regionally and/or internationally famous artistes in Singapore from 2008 - 2026

## External Dataset Cleaning and Validation

In [67]:
DATA_PATH = Path("../data/raw")

print(DATA_PATH.resolve())
print(DATA_PATH.exists())

/Users/gerald/Documents/stb-analytics-assessment/stb-analytics-assessment/data/raw
True


In [404]:
singapore_concert_calendar = pd.read_excel(
    DATA_PATH / "singapore_international_concert_calendar.xlsx"
)

In [ ]:
## visitor arrivals by country and age group and sex

country_demographic_monthly = connection.sql("""
    SELECT
        year,
        month,
        country,
        age_group,
        sex,
        SUM(visitor_arrivals) AS visitor_arrivals
    FROM length_of_stay_clean
    WHERE country IS NOT NULL
        AND age_group <> 'Not Stated'
        AND sex <> 'Not Stated'
    GROUP BY year, month, country, age_group, sex
    ORDER BY country, age_group, sex, year, month
""").df()

connection.register("country_demographic_monthly", country_demographic_monthly)
country_demographic_monthly

,year,month,country,age_group,sex,visitor_arrivals
0,2008,1,Australia,Age 14 & Below,Female,4473.0
1,2008,2,Australia,Age 14 & Below,Female,1321.0
2,2008,3,Australia,Age 14 & Below,Female,1728.0
3,2008,4,Australia,Age 14 & Below,Female,2596.0
4,2008,5,Australia,Age 14 & Below,Female,1558.0
...,...,...,...,...,...,...
177854,2026,3,Vietnam,Age 65 & Above,Male,721.0
177855,2026,4,Vietnam,Age 65 & Above,Male,666.0
177856,2026,5,Vietnam,Age 65 & Above,Male,760.0
177857,2026,6,Vietnam,Age 65 & Above,Male,863.0


In [ ]:
## add previous year's visitors arrivals to current year

country_demographic_yoy_base = connection.sql("""
    SELECT
        *,
        LAG(visitor_arrivals, 12) OVER (
            PARTITION BY country, age_group, sex
            ORDER BY year, month
        ) AS previous_year_visitors
    FROM country_demographic_monthly
""").df()

connection.register("country_demographic_yoy_base", country_demographic_yoy_base)
country_demographic_yoy_base

,year,month,country,age_group,sex,visitor_arrivals,previous_year_visitors
0,2018,5,Indonesia,Age 15 - 19,Female,5105.0,4326.0
1,2018,6,Indonesia,Age 15 - 19,Female,12258.0,9673.0
2,2018,7,Indonesia,Age 15 - 19,Female,8876.0,9382.0
3,2018,8,Indonesia,Age 15 - 19,Female,4428.0,4291.0
4,2018,9,Indonesia,Age 15 - 19,Female,4035.0,3666.0
...,...,...,...,...,...,...,...
177854,2019,7,Other Markets in Southeast Asia,Age 15 - 19,Male,273.0,263.0
177855,2019,8,Other Markets in Southeast Asia,Age 15 - 19,Male,261.0,222.0
177856,2019,9,Other Markets in Southeast Asia,Age 15 - 19,Male,275.0,198.0
177857,2019,10,Other Markets in Southeast Asia,Age 15 - 19,Male,198.0,234.0


In [ ]:
## add percentage change to yoy_arrivals

country_demographic_yoy = connection.sql("""
    SELECT
        year,
        month,
        country,
        age_group,
        sex,
        visitor_arrivals,
        ROUND(
            (visitor_arrivals / NULLIF(previous_year_visitors, 0) - 1) * 100,
            2
        ) AS visitor_yoy_pct
    FROM country_demographic_yoy_base
    WHERE previous_year_visitors IS NOT NULL
""").df()

connection.register("country_demographic_yoy", country_demographic_yoy)
country_demographic_yoy

,year,month,country,age_group,sex,visitor_arrivals,visitor_yoy_pct
0,2018,5,Indonesia,Age 15 - 19,Female,5105.0,18.01
1,2018,6,Indonesia,Age 15 - 19,Female,12258.0,26.72
2,2018,7,Indonesia,Age 15 - 19,Female,8876.0,-5.39
3,2018,8,Indonesia,Age 15 - 19,Female,4428.0,3.19
4,2018,9,Indonesia,Age 15 - 19,Female,4035.0,10.07
...,...,...,...,...,...,...,...
167678,2019,7,Other Markets in Southeast Asia,Age 15 - 19,Male,273.0,3.80
167679,2019,8,Other Markets in Southeast Asia,Age 15 - 19,Male,261.0,17.57
167680,2019,9,Other Markets in Southeast Asia,Age 15 - 19,Male,275.0,38.89
167681,2019,10,Other Markets in Southeast Asia,Age 15 - 19,Male,198.0,-15.38


In [ ]:
## remove covid years from analysis

country_demographic_yoy_clean = connection.sql("""
    SELECT *
    FROM country_demographic_yoy
    WHERE year NOT IN (2020, 2021, 2022, 2023)
""").df()

connection.register("country_demographic_yoy_clean", country_demographic_yoy_clean)
country_demographic_yoy_clean

,year,month,country,age_group,sex,visitor_arrivals,visitor_yoy_pct
0,2018,5,Indonesia,Age 15 - 19,Female,5105.0,18.01
1,2018,6,Indonesia,Age 15 - 19,Female,12258.0,26.72
2,2018,7,Indonesia,Age 15 - 19,Female,8876.0,-5.39
3,2018,8,Indonesia,Age 15 - 19,Female,4428.0,3.19
4,2018,9,Indonesia,Age 15 - 19,Female,4035.0,10.07
...,...,...,...,...,...,...,...
133238,2019,7,Other Markets in Southeast Asia,Age 15 - 19,Male,273.0,3.80
133239,2019,8,Other Markets in Southeast Asia,Age 15 - 19,Male,261.0,17.57
133240,2019,9,Other Markets in Southeast Asia,Age 15 - 19,Male,275.0,38.89
133241,2019,10,Other Markets in Southeast Asia,Age 15 - 19,Male,198.0,-15.38


In [ ]:
# create binary concert calendar indicator - list whether each month got concert vs no concert

concert_monthly_binary = connection.sql("""
    SELECT
        CAST(event_year AS INTEGER) AS year,
        CAST(event_month AS INTEGER) AS month,
        1 AS has_concert
    FROM singapore_concert_calendar
    GROUP BY event_year, event_month
    ORDER BY year, month
""").df()

connection.register(
    "concert_monthly_binary",
    concert_monthly_binary
)
concert_monthly_binary

,year,month,has_concert
0,2008,1,1
1,2008,2,1
2,2008,3,1
3,2008,5,1
4,2008,9,1
...,...,...,...
146,2025,11,1
147,2025,12,1
148,2026,4,1
149,2026,6,1


In [ ]:
# mesh tourism arrival data with binary concert calendar indicator

country_demographic_concert_binary = connection.sql("""
    SELECT
        d.year,
        d.month,
        d.country,
        d.age_group,
        d.sex,
        d.visitor_yoy_pct,
        COALESCE(c.has_concert, 0) AS has_concert
    FROM country_demographic_yoy_clean d
    LEFT JOIN concert_monthly_binary c
        ON d.year = c.year
        AND d.month = c.month
    ORDER BY d.country, d.age_group, d.sex, d.year, d.month
""").df()

connection.register(
    "country_demographic_concert_binary",
    country_demographic_concert_binary
)
country_demographic_concert_binary

,year,month,country,age_group,sex,visitor_yoy_pct,has_concert
0,2009,1,Australia,Age 14 & Below,Female,3.53,0
1,2009,2,Australia,Age 14 & Below,Female,-9.08,0
2,2009,3,Australia,Age 14 & Below,Female,-24.07,1
3,2009,4,Australia,Age 14 & Below,Female,24.42,1
4,2009,5,Australia,Age 14 & Below,Female,0.77,0
...,...,...,...,...,...,...,...
133238,2026,3,Vietnam,Age 65 & Above,Male,-7.68,0
133239,2026,4,Vietnam,Age 65 & Above,Male,-9.26,1
133240,2026,5,Vietnam,Age 65 & Above,Male,0.53,0
133241,2026,6,Vietnam,Age 65 & Above,Male,-7.10,1


In [458]:
# calculate uplift with vs without concert

country_demographic_binary_uplift = connection.sql("""
    SELECT
        country,
        age_group,
        sex,
        COUNT(*) FILTER (WHERE has_concert = 1) AS concert_months,
        COUNT(*) FILTER (WHERE has_concert = 0) AS no_concert_months,
        ROUND(
            AVG(visitor_yoy_pct) FILTER (WHERE has_concert = 1),
            2
        ) AS concert_month_avg_yoy,
        ROUND(
            AVG(visitor_yoy_pct) FILTER (WHERE has_concert = 0),
            2
        ) AS no_concert_month_avg_yoy,
        ROUND(
            AVG(visitor_yoy_pct) FILTER (WHERE has_concert = 1)
            - AVG(visitor_yoy_pct) FILTER (WHERE has_concert = 0),
            2
        ) AS concert_uplift_pct_point
    FROM country_demographic_concert_binary
    WHERE country NOT IN ('Others', 'Not Stated')
    GROUP BY country, age_group, sex
    ORDER BY concert_uplift_pct_point DESC
""").df()

connection.register(
    "country_demographic_binary_uplift",
    country_demographic_binary_uplift
)
country_demographic_binary_uplift

,country,age_group,sex,concert_months,no_concert_months,concert_month_avg_yoy,no_concert_month_avg_yoy,concert_uplift_pct_point
0,Kuwait,Age 15 - 19,Male,115,36,145.40,55.22,90.17
1,Iran,Age 15 - 19,Male,118,36,78.29,18.06,60.24
2,Mauritius,Age 15 - 19,Male,124,38,50.62,-3.29,53.91
3,Other Markets in North Asia,Age 25 - 34,Male,121,38,64.56,17.13,47.43
4,Israel,Age 14 & Below,Male,125,38,58.33,11.83,46.51
...,...,...,...,...,...,...,...,...
811,Kuwait,Age 20 - 24,Male,123,38,16.76,86.56,-69.80
812,Kuwait,Age 65 & Above,Female,120,34,45.77,121.81,-76.04
813,Egypt,Age 15 - 19,Male,111,32,94.03,180.31,-86.28
814,Other Markets in North Asia,Age 15 - 19,Female,68,22,16.22,108.07,-91.85


In [461]:
# t-test to check whether the difference between concert and no-concert months is statistically significant.

from scipy.stats import ttest_ind
import pandas as pd

binary_test_results = []

for (country, age, sex), temp in country_demographic_concert_binary.groupby(
    ["country", "age_group", "sex"]
):
    concert = temp.loc[
        temp["has_concert"] == 1,
        "visitor_yoy_pct"
    ].dropna()

    no_concert = temp.loc[
        temp["has_concert"] == 0,
        "visitor_yoy_pct"
    ].dropna()

    if len(concert) >= 30 and len(no_concert) >= 30:
        t_stat, p = ttest_ind(
            concert,
            no_concert,
            equal_var=False
        )

        binary_test_results.append({
            "country": country,
            "age_group": age,
            "sex": sex,
            "concert_months": len(concert),
            "no_concert_months": len(no_concert),
            "concert_avg_yoy": concert.mean(),
            "no_concert_avg_yoy": no_concert.mean(),
            "uplift_pct_point": concert.mean() - no_concert.mean(),
            "t_statistic": t_stat,
            "p_value": p
        })

binary_test_results = pd.DataFrame(binary_test_results)

connection.register(
    "binary_test_results",
    binary_test_results
)
binary_test_results

,country,age_group,sex,concert_months,no_concert_months,concert_avg_yoy,no_concert_avg_yoy,uplift_pct_point,t_statistic,p_value
0,Australia,Age 14 & Below,Female,125,38,6.65992,9.050263,-2.390343,-0.824205,0.413180
1,Australia,Age 14 & Below,Male,125,38,6.82576,9.350263,-2.524503,-0.944810,0.348371
2,Australia,Age 15 - 19,Female,125,38,5.53616,4.350000,1.186160,0.337536,0.736738
3,Australia,Age 15 - 19,Male,125,38,5.68560,2.298421,3.387179,1.145579,0.255274
4,Australia,Age 20 - 24,Female,125,38,3.96920,-1.825263,5.794463,2.255509,0.027596
...,...,...,...,...,...,...,...,...,...,...
805,Vietnam,Age 45 - 54,Male,125,38,1.82352,8.977632,-7.154112,-2.484045,0.016273
806,Vietnam,Age 55 - 64,Female,125,38,8.11192,22.381316,-14.269396,-2.931450,0.005175
807,Vietnam,Age 55 - 64,Male,125,38,4.85768,15.825000,-10.967320,-3.400809,0.001246
808,Vietnam,Age 65 & Above,Female,125,38,8.94024,18.143421,-9.203181,-2.000775,0.050651


In [466]:
# view all statistically significant results

significant_binary_results = connection.sql("""
    SELECT
        country,
        age_group,
        sex,
        concert_months,
        no_concert_months,
        ROUND(concert_avg_yoy, 2) AS concert_avg_yoy,
        ROUND(no_concert_avg_yoy, 2) AS no_concert_avg_yoy,
        ROUND(uplift_pct_point, 2) AS uplift_pct_point,
        ROUND(t_statistic, 3) AS t_statistic,
        ROUND(p_value, 4) AS p_value
    FROM binary_test_results
    WHERE p_value < 0.05
        AND uplift_pct_point > 0
    ORDER BY uplift_pct_point DESC
""").df()

connection.register(
    "significant_binary_results",
    significant_binary_results
)

significant_binary_results

,country,age_group,sex,concert_months,no_concert_months,concert_avg_yoy,no_concert_avg_yoy,uplift_pct_point,t_statistic,p_value
0,Mauritius,Age 15 - 19,Male,123,37,50.62,-3.29,53.91,2.038,0.0433
1,Israel,Age 14 & Below,Male,125,38,58.33,11.83,46.51,2.638,0.0092
2,Rep of Ireland,Age 15 - 19,Male,123,38,53.26,8.27,44.99,2.260,0.0253
3,Israel,Age 14 & Below,Female,125,38,53.89,9.89,43.99,2.679,0.0083
4,Pakistan,Age 15 - 19,Male,125,38,40.15,1.76,38.39,2.396,0.0177
...,...,...,...,...,...,...,...,...,...,...
123,USA,Age 55 - 64,Female,125,38,9.70,5.27,4.43,2.165,0.0343
124,Other Markets in Europe,Age 45 - 54,Male,125,38,10.80,6.52,4.27,2.087,0.0403
125,France,Age 25 - 34,Female,125,38,7.64,3.48,4.17,2.180,0.0325
126,USA,Age 25 - 34,Female,125,38,8.26,4.41,3.85,2.117,0.0381


In [ ]:
# summarize binary results by age and sex

positive_significant_binary_by_age_sex = connection.sql("""
    SELECT
        age_group,
        sex,
        COUNT(*) AS total_tests,
        COUNT(*) FILTER (
            WHERE p_value < 0.05
                AND uplift_pct_point > 0
        ) AS positive_significant_results,
        ROUND(
            COUNT(*) FILTER (
                WHERE p_value < 0.05
                    AND uplift_pct_point > 0
            ) * 100.0 / COUNT(*),
            2
        ) AS positive_significant_pct,
        ROUND(
            AVG(uplift_pct_point) FILTER (
                WHERE p_value < 0.05
                    AND uplift_pct_point > 0
            ),
            2
        ) AS avg_significant_uplift_pct_point
    FROM binary_test_results
    GROUP BY age_group, sex
    ORDER BY positive_significant_pct DESC
""").df()

connection.register(
    "positive_significant_binary_by_age_sex",
    positive_significant_binary_by_age_sex
)

positive_significant_binary_by_age_sex

## Of the 50–51 source countries tested, the most widespread statistically significant positive association 
# between concert months and YoY visitor growth was observed among male tourists aged 15–44, as well as 
# female tourists aged 25–34.


,age_group,sex,total_tests,positive_significant_results,positive_significant_pct,avg_significant_uplift_pct_point
0,Age 35 - 44,Male,51,13,25.49,7.60
1,Age 25 - 34,Male,51,13,25.49,8.40
2,Age 15 - 19,Male,50,11,22.00,26.80
3,Age 20 - 24,Male,51,11,21.57,17.15
4,Age 25 - 34,Female,51,11,21.57,9.95
5,Age 35 - 44,Female,51,9,17.65,9.43
6,Age 20 - 24,Female,50,8,16.00,12.91
7,Age 14 & Below,Male,50,8,16.00,16.25
8,Age 45 - 54,Female,51,8,15.69,9.85
9,Age 55 - 64,Male,51,7,13.73,6.79


In [471]:
# create concert calendar indicator - list no of concerts each month

concert_monthly = connection.sql("""
    SELECT
        CAST(event_year AS INTEGER) AS year,
        CAST(event_month AS INTEGER) AS month,
        COUNT(*) AS concert_count
    FROM singapore_concert_calendar
    GROUP BY event_year, event_month
    ORDER BY year, month
""").df()

connection.register("concert_monthly", concert_monthly)
concert_monthly

,year,month,concert_count
0,2008,1,1
1,2008,2,1
2,2008,3,2
3,2008,5,1
4,2008,9,1
...,...,...,...
146,2025,11,4
147,2025,12,1
148,2026,4,1
149,2026,6,2


In [ ]:
## check that 340 rows still present

connection.sql("""
    SELECT
        SUM(concert_count) AS total_concerts,
        COUNT(*) AS concert_months
    FROM concert_monthly
""").df()

,total_concerts,concert_months
0,340.0,151


In [473]:
# mesh tourism arrival data with concert calendar indicator

country_demographic_concert = connection.sql("""
    SELECT
        d.year,
        d.month,
        d.country,
        d.age_group,
        d.sex,
        d.visitor_yoy_pct,
        COALESCE(c.concert_count, 0) AS concert_count,
        CASE
            WHEN COALESCE(c.concert_count, 0) > 0 THEN 1
            ELSE 0
        END AS has_concert
    FROM country_demographic_yoy_clean d
    LEFT JOIN concert_monthly c
        ON d.year = c.year
        AND d.month = c.month
    WHERE d.country NOT IN ('Not Stated', 'Others')
    ORDER BY d.country, d.age_group, d.sex, d.year, d.month
""").df()

connection.register(
    "country_demographic_concert",
    country_demographic_concert
)
country_demographic_concert

,year,month,country,age_group,sex,visitor_yoy_pct,concert_count,has_concert
0,2009,1,Australia,Age 14 & Below,Female,3.53,0,0
1,2009,2,Australia,Age 14 & Below,Female,-9.08,0,0
2,2009,3,Australia,Age 14 & Below,Female,-24.07,1,1
3,2009,4,Australia,Age 14 & Below,Female,24.42,1,1
4,2009,5,Australia,Age 14 & Below,Female,0.77,0,0
...,...,...,...,...,...,...,...,...
132352,2026,3,Vietnam,Age 65 & Above,Male,-7.68,0,0
132353,2026,4,Vietnam,Age 65 & Above,Male,-9.26,1,1
132354,2026,5,Vietnam,Age 65 & Above,Male,0.53,0,0
132355,2026,6,Vietnam,Age 65 & Above,Male,-7.10,2,1


In [476]:
# calculate correlation between concert count and visitor arrivals change yoy

country_demographic_correlation = connection.sql("""
    SELECT
        country,
        age_group,
        sex,
        ROUND(
            CORR(concert_count, visitor_yoy_pct),
            3
        ) AS correlation,
        COUNT(*) AS observations
    FROM country_demographic_concert
    WHERE visitor_yoy_pct IS NOT NULL
    GROUP BY country, age_group, sex
    HAVING COUNT(*) >= 30
    ORDER BY correlation DESC
""").df()

connection.register(
    "country_demographic_correlation",
    country_demographic_correlation
)

country_demographic_correlation

,country,age_group,sex,correlation,observations
0,UK,Age 35 - 44,Male,0.409,163
1,UK,Age 25 - 34,Male,0.346,163
2,UK,Age 35 - 44,Female,0.339,163
3,United Arab Emirates,Age 45 - 54,Male,0.339,163
4,China,Age 20 - 24,Male,0.330,163
...,...,...,...,...,...
811,Vietnam,Age 45 - 54,Male,-0.283,163
812,Vietnam,Age 35 - 44,Female,-0.299,163
813,Vietnam,Age 45 - 54,Female,-0.319,163
814,Vietnam,Age 55 - 64,Female,-0.352,163


In [ ]:
## correlation test

from scipy.stats import pearsonr
import pandas as pd

country_demographic_pvalues = []

for (country, age, sex), temp in country_demographic_concert.groupby(
    ["country", "age_group", "sex"]
):
    temp = temp.dropna(
        subset=["concert_count", "visitor_yoy_pct"]
    )

    if len(temp) >= 30:
        r, p = pearsonr(
            temp["concert_count"],
            temp["visitor_yoy_pct"]
        )

        country_demographic_pvalues.append({
            "country": country,
            "age_group": age,
            "sex": sex,
            "correlation": r,
            "p_value": p,
            "observations": len(temp)
        })

country_demographic_pvalues = pd.DataFrame(
    country_demographic_pvalues
)

connection.register(
    "country_demographic_pvalues",
    country_demographic_pvalues
)

country_demographic_pvalues

,country,age_group,sex,correlation,p_value,observations
0,Australia,Age 14 & Below,Female,-0.025232,0.749191,163
1,Australia,Age 14 & Below,Male,-0.006006,0.939343,163
2,Australia,Age 15 - 19,Female,0.108517,0.167936,163
3,Australia,Age 15 - 19,Male,0.180341,0.021242,163
4,Australia,Age 20 - 24,Female,0.227554,0.003486,163
...,...,...,...,...,...,...
811,Vietnam,Age 45 - 54,Male,-0.282959,0.000252,163
812,Vietnam,Age 55 - 64,Female,-0.351932,0.000004,163
813,Vietnam,Age 55 - 64,Male,-0.357819,0.000003,163
814,Vietnam,Age 65 & Above,Female,-0.235877,0.002436,163


In [ ]:
## mark results by significant positive and negative

final_intensity_concert_analysis = connection.sql("""
    SELECT
        country,
        age_group,
        sex,
        ROUND(correlation, 3) AS correlation,
        ROUND(p_value, 4) AS p_value,
        observations,
        CASE
            WHEN p_value < 0.05 AND correlation > 0
                THEN 'Significant Positive'
            WHEN p_value < 0.05 AND correlation < 0
                THEN 'Significant Negative'
            ELSE 'Not Significant'
        END AS result
    FROM country_demographic_pvalues
    ORDER BY p_value ASC
""").df()

connection.register(
    "final_intensity_concert_analysis",
    final_intensity_concert_analysis
)

final_intensity_concert_analysis

,country,age_group,sex,correlation,p_value,observations,result
0,China,Age 20 - 24,Male,0.330,0.0000,163,Significant Positive
1,Other Markets in Africa,Age 25 - 34,Male,0.324,0.0000,163,Significant Positive
2,Other Markets in West Asia,Age 20 - 24,Male,0.314,0.0000,163,Significant Positive
3,Rep of Ireland,Age 25 - 34,Female,0.327,0.0000,163,Significant Positive
4,UK,Age 20 - 24,Female,0.321,0.0000,163,Significant Positive
...,...,...,...,...,...,...,...
811,Malaysia,Age 35 - 44,Male,0.001,0.9914,163,Not Significant
812,Norway,Age 65 & Above,Male,0.001,0.9923,163,Not Significant
813,Norway,Age 20 - 24,Male,0.000,0.9962,163,Not Significant
814,South Korea,Age 25 - 34,Female,-0.000,0.9980,163,Not Significant


In [ ]:
## view significant positive results only

significant_positive_intensity = connection.sql("""
    SELECT *
    FROM final_intensity_concert_analysis
    WHERE result = 'Significant Positive'
    ORDER BY correlation DESC
""").df()

connection.register(
    "significant_positive_intensity",
    significant_positive_intensity
)

significant_positive_intensity

,country,age_group,sex,correlation,p_value,observations,result
0,UK,Age 35 - 44,Male,0.409,0.0000,163,Significant Positive
1,UK,Age 25 - 34,Male,0.346,0.0000,163,Significant Positive
2,UK,Age 35 - 44,Female,0.339,0.0000,163,Significant Positive
3,United Arab Emirates,Age 45 - 54,Male,0.339,0.0000,163,Significant Positive
4,China,Age 20 - 24,Male,0.330,0.0000,163,Significant Positive
...,...,...,...,...,...,...,...
176,China,Age 45 - 54,Female,0.157,0.0450,163,Significant Positive
177,Spain,Age 14 & Below,Female,0.157,0.0455,163,Significant Positive
178,Philippines,Age 25 - 34,Female,0.156,0.0468,163,Significant Positive
179,India,Age 25 - 34,Female,0.155,0.0487,163,Significant Positive


In [ ]:
# relative ranking of statistically significant positive association results by age group and sex

positive_significant_intensity_by_age_sex = connection.sql("""
    SELECT
        age_group,
        sex,
        COUNT(*) AS total_tests,
        COUNT(*) FILTER (
            WHERE p_value < 0.05
                AND correlation > 0
        ) AS positive_significant_results,
        ROUND(
            COUNT(*) FILTER (
                WHERE p_value < 0.05
                    AND correlation > 0
            ) * 100.0 / COUNT(*),
            2
        ) AS positive_significant_pct,
        ROUND(
            AVG(correlation) FILTER (
                WHERE p_value < 0.05
                    AND correlation > 0
            ),
            3
        ) AS avg_significant_correlation
    FROM country_demographic_pvalues
    GROUP BY age_group, sex
    ORDER BY positive_significant_pct DESC
""").df()

connection.register(
    "positive_significant_intensity_by_age_sex",
    positive_significant_intensity_by_age_sex
)

positive_significant_intensity_by_age_sex

## Of the 51 source countries tested, the most widespread statistically significant 
# positive correlations between concert intensity and YoY visitor growth were observed 
# among male tourists aged 25–64, as well as female tourists aged 45–54.


,age_group,sex,total_tests,positive_significant_results,positive_significant_pct,avg_significant_correlation
0,Age 35 - 44,Male,51,19,37.25,0.236
1,Age 25 - 34,Male,51,18,35.29,0.229
2,Age 45 - 54,Male,51,17,33.33,0.231
3,Age 55 - 64,Male,51,16,31.37,0.204
4,Age 45 - 54,Female,51,16,31.37,0.203
5,Age 20 - 24,Male,51,14,27.45,0.247
6,Age 20 - 24,Female,51,13,25.49,0.223
7,Age 25 - 34,Female,51,11,21.57,0.215
8,Age 35 - 44,Female,51,11,21.57,0.225
9,Age 15 - 19,Male,51,11,21.57,0.198


In [ ]:
## how many top 10 source tourism markets appeared in the results (concert vs no concert)

top10_positive_binary_count = connection.sql("""
    SELECT
        COUNT(DISTINCT country) FILTER (
            WHERE country IN (
                SELECT country
                FROM average_country_rank_top10
            )
        ) AS top10_markets_represented
    FROM binary_test_results
    WHERE p_value < 0.05
        AND uplift_pct_point > 0
""").df()

top10_positive_binary_count

,top10_markets_represented
0,6


In [ ]:
## how many top 10 source tourism markets appeared in the results (more concerts = more tourist arrivals)

top10_positive_intensity_count = connection.sql("""
    SELECT
        COUNT(DISTINCT country) FILTER (
            WHERE country IN (
                SELECT country
                FROM average_country_rank_top10
            )
        ) AS top10_markets_represented
    FROM country_demographic_pvalues
    WHERE p_value < 0.05
        AND correlation > 0
""").df()

top10_positive_intensity_count

,top10_markets_represented
0,7


# (3) Insights Generation

First, when comparing visitor growth during months with international concerts against months without concerts, 6 of Singapore's top 10 tourist markets exhibited statistically significant positive uplift (p < 0.05) among the target demographic groups.

Second, when examining whether months with a greater number of international concerts were associated with stronger visitor growth, 7 of the top 10 tourist markets exhibited statistically significant positive correlations (p < 0.05) among the target demographic groups.

Combined, these results suggest that international concerts are associated with stronger visitor growth among the
target demographics across a majority of Singapore's largest tourism source markets. The intensity result further suggests that the relationship is not limited to whether a concert occurred: in 7 of the 10 top markets, greater concert activity was positively associated with stronger YoY visitor growth for at least one target demographic.

That said, these findings do not establish that concerts caused these demographics to constitute a higher proportion of visitors. Other factors like seasonality, holidays, other major events in Singapore and broader economic conditions in the source markets may also influence the age group makeup of tourist arrivals from Singapore's top 10 tourism source markets.

All-in-all however, these data findings suggest that greater international concert activity in Singapore is associated with stronger tourist arrival growth among our largest demographic groups across a majority of our top 10 tourism source markets.
